# Fine-tuning Faster R-CNN cho ISIC 2018 Task 1
Dùng ảnh soi da để học phát hiện một lớp `skin_lesion` bằng box trích từ mask ground truth. Ba thí nghiệm dùng chung split nội bộ từ bộ training chính thức.

## 1. Thông tin môi trường
Chọn **Runtime → Change runtime type → GPU**. Chạy kiểm tra trong một tiến trình riêng để phiên notebook có thể cài lại wheel nếu cần.

In [ ]:
import importlib.metadata as metadata
import shutil
import subprocess
import sys

print(f"Python {sys.version.split()[0]}")
for package in ("torch", "torchvision"):
    try:
        print(f"{package} {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: chưa cài")
print(f"Dung lượng trống: {shutil.disk_usage('/content').free / 2**30:.1f} GiB")
probe = subprocess.run([sys.executable, "-c", """
import torch
print('CUDA khả dụng:', torch.cuda.is_available(), '| CUDA build:', torch.version.cuda)
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'{gpu.name} | VRAM {gpu.total_memory / 2**30:.2f} GiB')
"""], capture_output=True, text=True)
print(probe.stdout if probe.returncode == 0 else probe.stderr)

## 2. Cài thư viện
Giữ bản PyTorch và Torchvision có sẵn nếu kiểm tra CUDA, AMP và NMS thành công. Bật `INSTALL_FALLBACK` nếu cần cài cặp wheel CUDA 12.6. Cấu hình và thư viện khi tiếp tục cần khớp với checkpoint.

In [ ]:
INSTALL_FALLBACK = False # @param {type:"boolean"}
gpu_check = subprocess.run(["nvidia-smi", "-L"], check=False, capture_output=True, text=True)
if gpu_check.returncode != 0:
    raise RuntimeError("Chọn GPU trong Runtime > Change runtime type.")
runtime_probe = """
import torch
import torchvision
device = torch.device('cuda:0')
assert torch.cuda.is_available()
assert callable(torch.amp.GradScaler)
with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
    assert torchvision.ops.nms(torch.zeros((1, 4), device=device), torch.ones(1, device=device), .5).item() == 0
"""
def check_runtime():
    result = subprocess.run([sys.executable, "-c", runtime_probe], capture_output=True, text=True)
    if result.returncode:
        print(result.stderr)
    return result.returncode == 0

if not check_runtime():
    if not INSTALL_FALLBACK:
        raise RuntimeError("Torch/Torchvision CUDA không tương thích. Bật INSTALL_FALLBACK rồi chạy lại ô.")
    subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.9.1", "torchvision==0.24.1",
                    "torchaudio==2.9.1", "--index-url", "https://download.pytorch.org/whl/cu126"], check=True)
    if not check_runtime():
        raise RuntimeError("Wheel CUDA chưa hoạt động. Kiểm tra GPU, driver và phiên bản Python của runtime.")
    if "torch" in sys.modules or "torchvision" in sys.modules:
        raise RuntimeError("Khởi động lại session rồi chạy lại từ ô 1 để nạp wheel mới.")

subprocess.run([sys.executable, "-m", "pip", "install",
                "numpy>=1.26,<3", "Pillow>=10,<13", "PyYAML>=6,<7", "tqdm>=4.66,<5",
                "matplotlib>=3.8,<4", "pycocotools>=2.0.8,<3"], check=True)
subprocess.run([sys.executable, "-c", "import pycocotools.mask"], check=True)
print("CUDA NMS, AMP và pycocotools sẵn sàng.")

## 3. Gắn Google Drive
Đặt ZIP của workspace tại `MyDrive/FasterRCNN_ISIC2018/finetune_detection.zip`. ZIP của ISIC có thể đặt trong thư mục `archives/`.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/FasterRCNN_ISIC2018") # @param {type:"string"}
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
ARCHIVES = DRIVE_ROOT / "archives"
PERSISTENT_MANIFEST = DRIVE_ROOT / "train_manifest.json"
print(DRIVE_ROOT)

## 4. Khởi tạo workspace
Dùng ZIP có thư mục cấp cao nhất `finetune_detection/`. Nếu đã giải nén, điền đường dẫn vào `EXTRACTED_WORKSPACE`.

In [ ]:
import hashlib
import os
import tempfile
from zipfile import ZipFile

EXTRACTED_WORKSPACE = "" # @param {type:"string"}
ZIP_NAME = "finetune_detection.zip" # @param {type:"string"}
LOCAL_ROOT = Path("/content/faster_rcnn_isic2018")
LOCAL_ROOT.mkdir(exist_ok=True)
RUNS_ROOT = LOCAL_ROOT / "runs"
DATA_ROOT = LOCAL_ROOT / "data"
if EXTRACTED_WORKSPACE:
    WORKSPACE = Path(EXTRACTED_WORKSPACE).resolve()
else:
    archive = DRIVE_ROOT / ZIP_NAME
    if not archive.is_file():
        raise FileNotFoundError(f"Tải workspace ZIP lên {archive}")
    fingerprint = hashlib.sha256(archive.read_bytes()).hexdigest()
    bundle_root = LOCAL_ROOT / f"workspace-{fingerprint}"
    WORKSPACE = bundle_root / "finetune_detection"
    if not WORKSPACE.is_dir():
        with ZipFile(archive) as zipped, tempfile.TemporaryDirectory(dir=LOCAL_ROOT) as temporary:
            stage = Path(temporary)
            for member in zipped.infolist():
                name = member.filename
                path = Path(name)
                if (not path.parts or path.parts[0] != "finetune_detection"
                        or ".." in path.parts or chr(92) in name or ":" in name):
                    raise ValueError(f"Thành viên ZIP không hợp lệ: {name}")
                zipped.extract(member, stage)
            stage.rename(bundle_root)
required = ["configs/E1.yaml", "configs/E2.yaml", "configs/E3.yaml", "src/trainer.py",
            "scripts/prepare_data.py", "requirements.txt"]
if missing := [relative for relative in required if not (WORKSPACE / relative).is_file()]:
    raise FileNotFoundError(f"Thiếu tệp workspace: {missing}")
os.chdir(WORKSPACE)
if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))
print(WORKSPACE)

## 5. Chuẩn bị dữ liệu huấn luyện
Các ZIP có sẵn được sao chép sang ổ đĩa của runtime. Bật tải ISIC để lấy tệp còn thiếu. Chính sách mặc định dừng khi gặp dữ liệu lỗi và ghi danh sách để kiểm tra.

In [ ]:
from scripts.prepare_data import prepare_dataset, persist_manifest
from src.dataset import load_manifest

DOWNLOAD_IF_MISSING = True # @param {type:"boolean"}
GROUPING_CSV = "" # @param {type:"string"}
INVALID_POLICY = "fail" # @param ["fail", "exclude"]
manifest_path = prepare_dataset(DATA_ROOT / "training", ARCHIVES, DOWNLOAD_IF_MISSING,
                                GROUPING_CSV or None, INVALID_POLICY)
persist_manifest(manifest_path, PERSISTENT_MANIFEST)
manifest = load_manifest(manifest_path)
print(f"Đã lưu manifest {manifest['sha256']} vào Drive.")

## 6. Kiểm tra dữ liệu
Histogram dùng diện tích box chia cho diện tích ảnh gốc. Khung xanh là box tính từ mask ground truth.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from src.dataset import LesionDataset

print(f"Ảnh nguồn {manifest['image_count']} | Hợp lệ {len(manifest['records'])} | "
      f"Lỗi {len(manifest['invalid'])} | Có lesion ID {manifest['group_coverage']}")
print({name: len(ids) for name, ids in manifest["splits"].items()})
fractions = [row["area"] / (row["width"] * row["height"]) for row in manifest["records"]]
plt.figure(figsize=(7, 3))
plt.hist(fractions, bins=20)
plt.xlabel("Tỷ lệ diện tích box / ảnh")
plt.ylabel("Số ảnh")
plt.show()
dataset = LesionDataset(manifest_path, manifest, "train")
count = min(4, len(dataset))
figure, axes = plt.subplots(1, count, figsize=(4 * count, 4), squeeze=False)
for i, axis in enumerate(axes[0]):
    index = i * len(dataset) // count
    image, target = dataset[index]
    x1, y1, x2, y2 = target["boxes"][0].tolist()
    axis.imshow(image.permute(1, 2, 0))
    axis.add_patch(Rectangle((x1 - 0.5, y1 - 0.5), x2 - x1, y2 - y1, fill=False, edgecolor="lime", linewidth=2))
    axis.set_title(dataset.records[index]["id"])
    axis.axis("off")
plt.show()

## 7. Chọn experiment
`ALL` chạy E1, E2 rồi E3 trên cùng một GPU, tiếp tục run dở và bỏ qua run đã hoàn tất. Cấu hình sửa đổi dùng `CONFIG_PATH` và `RUN_NAME` riêng.

In [ ]:
from src.utils import EXPERIMENTS, load_config

EXPERIMENT = "E1" # @param ["E1", "E2", "E3", "ALL"]
CONFIG_PATH = "" # @param {type:"string"}
RUN_NAME = "" # @param {type:"string"}
if CONFIG_PATH and not RUN_NAME:
    raise ValueError("Điền RUN_NAME để giữ riêng cấu hình tùy chỉnh.")
if EXPERIMENT == "ALL" and (CONFIG_PATH or RUN_NAME):
    raise ValueError("ALL chạy ba cấu hình mặc định.")
for name in EXPERIMENTS if EXPERIMENT == "ALL" else (EXPERIMENT,):
    print(load_config(name, CONFIG_PATH or None))

## 8. Huấn luyện và theo dõi trực tiếp
Mỗi epoch có thanh Training và Validation ngay dưới ô này. Sau khi Colab ngắt kết nối, mở lại notebook, giữ nguyên cấu hình và chạy tới ô này để tiếp tục từ checkpoint đã xác minh.

In [ ]:
from src.trainer import run_all_experiments, run_experiment

options = dict(manifest_path=manifest_path, runs_root=RUNS_ROOT, drive_root=DRIVE_ROOT, resume="auto")
if EXPERIMENT == "ALL":
    results = run_all_experiments(**options)
else:
    results = {EXPERIMENT: run_experiment(EXPERIMENT, config_path=CONFIG_PATH or None, run_name=RUN_NAME or None, **options)}

## 9. Xem kết quả validation
Chọn experiment để xem chỉ số tại best epoch và so sánh box dự đoán với ground truth. Ngưỡng score lấy từ cấu hình đã lưu.

In [ ]:
from pprint import pprint
from src.checkpoint import CheckpointStore
from src.evaluator import preview_validation
from src.trainer import result_summary
from src.utils import require_device, run_directory

VIEW_EXPERIMENT = "E1" # @param ["E1", "E2", "E3"]
def view_store(experiment, run_name=None):
    return CheckpointStore(run_directory(RUNS_ROOT, experiment, run_name),
                           run_directory(DRIVE_ROOT, experiment, run_name))

view_name = RUN_NAME or None if VIEW_EXPERIMENT == EXPERIMENT else None
store = view_store(VIEW_EXPERIMENT, view_name)
latest = store.latest()
if latest and latest["progress"]["best_epoch"] is not None:
    pprint(result_summary(latest["progress"], latest))
    preview_validation(store.best(latest), manifest_path, require_device())
    del latest
else:
    print("Chưa có epoch validation để hiển thị.")

## 10. So sánh E1, E2 và E3
Bảng dùng chung manifest validation. Xem Recall và FP/Image bên cạnh mAP trước khi chọn model để triển khai.

In [ ]:
from IPython.display import Markdown, display
from scripts.compare_experiments import compare, render_report

rows, candidate = compare(RUNS_ROOT, DRIVE_ROOT)
report = render_report(rows, candidate)
display(Markdown(report))
if rows:
    (DRIVE_ROOT / "comparison.md").write_text(report + "\n", encoding="utf-8")

## 11. Xuất inference checkpoint
Điền experiment đã chọn sau khi xem validation. Ô này lưu weights và metadata tiền xử lý kèm run ID. Quy trình đánh giá official test nằm trong README.

In [ ]:
import tempfile
from src.annotations import sha256_file
from src.evaluator import export_inference

EXPORT_EXPERIMENT = "" # @param ["", "E1", "E2", "E3"]
EXPORT_RUN_NAME = "" # @param {type:"string"}
if EXPORT_EXPERIMENT:
    store = view_store(EXPORT_EXPERIMENT, EXPORT_RUN_NAME or None)
    latest = store.latest()
    if not latest or not latest["progress"]["completed"]:
        raise ValueError("Chỉ export run đã hoàn tất và có best checkpoint hợp lệ.")
    best = store.best(latest)
    destination = DRIVE_ROOT / "exports" / f"{EXPORT_EXPERIMENT}-{latest['run_id']}.pt"
    if destination.exists():
        from src.model import load_inference
        model, existing = load_inference(destination)
        if existing["selection"]["run_id"] != latest["run_id"]:
            raise ValueError(f"Artifact Drive khác run đã chọn: {destination}")
        del model, existing
    else:
        destination.parent.mkdir(parents=True, exist_ok=True)
        with tempfile.TemporaryDirectory(dir=LOCAL_ROOT) as temporary:
            local_artifact = export_inference(best, latest, Path(temporary) / destination.name)
            staged = destination.with_suffix(".pt.part")
            try:
                shutil.copyfile(local_artifact, staged)
                checksum = sha256_file(local_artifact)
                if sha256_file(staged) != checksum:
                    raise OSError("SHA-256 của bản sao Drive không khớp.")
                staged.replace(destination)
                if sha256_file(destination) != checksum:
                    raise OSError("Không xác minh được artifact xuất lên Drive.")
            finally:
                staged.unlink(missing_ok=True)
    print(destination)
    del latest, best
else:
    print("Chọn EXPORT_EXPERIMENT khi đã chốt model bằng validation.")